## Loading dataset & Preprocessing

#### Import libraries

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [15]:

training_data = pd.read_csv('Network Intrusion Detection- UNSW_NB15 training and testing sets/UNSW_NB15_training-set.csv',
                            sep=",")
testing_set = pd.read_csv('Network Intrusion Detection- UNSW_NB15 training and testing sets/UNSW_NB15_testing-set.csv', 
                          sep=",")

columns = training_data.columns
attacks = training_data.query("label == 1") 
no_services = training_data.query("service == '-'")

#print("total features: ")
#print(columns)

#print(no_services)
#print(attacks)

print("full dataset(unchanged): ")
print(training_data)



full dataset(unchanged): 
            id       dur proto service state  spkts  dpkts  sbytes  dbytes  \
0            1  0.121478   tcp       -   FIN      6      4     258     172   
1            2  0.649902   tcp       -   FIN     14     38     734   42014   
2            3  1.623129   tcp       -   FIN      8     16     364   13186   
3            4  1.681642   tcp     ftp   FIN     12     12     628     770   
4            5  0.449454   tcp       -   FIN     10      6     534     268   
...        ...       ...   ...     ...   ...    ...    ...     ...     ...   
175336  175337  0.000009   udp     dns   INT      2      0     114       0   
175337  175338  0.505762   tcp       -   FIN     10      8     620     354   
175338  175339  0.000009   udp     dns   INT      2      0     114       0   
175339  175340  0.000009   udp     dns   INT      2      0     114       0   
175340  175341  0.000009   udp     dns   INT      2      0     114       0   

                 rate  ...  ct_dst_sp

### One-Hot encoding: protocols, states, services

In [19]:

altered_training = training_data.drop(["id","attack_cat", "label"], axis=1)

""" quick test
print("Altered dataset: ")
print(altered_training)
"""

categorical_col = ['proto', 'service', 'state']
numerical_col = altered_training.drop(columns=categorical_col).columns

print("numerical colums in our dataset: ")
print(numerical_col)

"""
protocols = training_data["proto"].unique() # 133 different protocols(including "none")
states = training_data["state"].unique() # 9 states(inlcuding "no"
services = training_data["service"].unique() # 13 services(including "-")
"""

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_col
        ),
        ("numerical", "passthrough", numerical_col)
    ],
    verbose_feature_names_out=False
)


numerical colums in our dataset: 
Index(['dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl',
       'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit',
       'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean',
       'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src',
       'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm',
       'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd',
       'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports'],
      dtype='str')


## Feature Selection methods

### SelectKBest 

In [18]:
"""
We should expect a total of 194 feature names that the pipeline handles
39 features(proto, state, and service excluded) from before, 133 unique protocols, 9 states, and 13 services
"""


feature_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("selection", SelectKBest(score_func=f_classif, k=15))
])

y_train = training_data["label"]
X_train = altered_training

feature_pipeline.fit_transform(X_train, y_train)
feature_names = feature_pipeline.named_steps["preprocessing"].get_feature_names_out()

selected = feature_pipeline.named_steps["selection"]
indices = selected.get_support()

scores = pd.DataFrame({
    "Feature": feature_names[indices],  
    "Score": selected.scores_[indices]}
)

scores = scores.sort_values(by="Score", ascending=False)
scores.index = pd.RangeIndex(start=1, stop=len(scores)+1, step=1)

print("Top 15 Features: ")
print(scores)


Top 15 Features: 
             Feature          Score
1               sttl  161780.526952
2       ct_state_ttl   87830.779493
3          state_INT   59703.332860
4              dload   32170.312851
5          state_CON   27258.672011
6   ct_dst_sport_ltm   25645.955324
7              dmean   23195.039749
8               rate   22611.817828
9          proto_tcp   21975.518040
10              swin   21961.796620
11              dwin   19950.909669
12  ct_src_dport_ltm   18059.210546
13    ct_dst_src_ltm   17835.399757
14         state_FIN   17765.078288
15             stcpb   12194.973688
